# Tutorial FluorophoreSystem

### Naming convention
|name|meaning|
|---|---|
|single state|photophysical state|
|joined state|combinations of single states for each fluorophore|
|single transition|change in single state|
|joined transition|change in joined state|
|emission|fluorescent emission|
|event|detected emission|
|resample / deltat|frame integration time, $\tau_{min}$, lag time$_{min}$|

### Handling of multiple possible transitions
Example: The transition between first excited state $S_1$ and ground state $S_0$ can be via fluorescence or via internal/external conversion followed by vibrational relaxation.

Strategy: 
- effected transition rates are added in initialize.construct_transition_matrices
- the rate used in the algorithm is hence dependent on both transitions
- to later discriminate between the two transitions:
    - process.multiple_transitions to create cumulative sum of the effected transitions
    - process.generate_transition_series inserts random number into cumulative sum

In [11]:
user = r"\SagixOffice"  # HomeOffice
#user = r"\vie43sq"  # University
import sys
sys.path.append(rf"C:\Users{user}\OneDrive - Universität Würzburg\GitHub\Photoswitching")

import numpy as np
import pandas as pd
import src.fluorophore_systems as fs
import src.miscellaneous as mi

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
rates = [['k_S0_S1', 7e6, "excitation", "EXC", False],  
         ['k_S1_S0', 1e9, "fluorescent emission", "FLU", True], 
         ['k_S1_T1', 1e6, "intersystem crossing ST", "ISCST", False],   
         ['k_T1_S0', 5e5, "intersystem crossing TS", "ISCTS", False],
         ['k_S1_S0', 1e9, "internal conversion S", "ICS", False]]

## Initialize object

In [52]:
system = fs.GeneralModel(number_fluorophores=2, distances=1, rates=rates)

In [53]:
system.single_states

{0: 'S0', 1: 'S1', 2: 'T1', 3: 'B'}

In [54]:
system.single_transitions

,name,rate,trivial_name,abbreviation,fluorescence
id,,,,,
0,k_S0_S1,7.000000e+06,excitation,EXC,False
1,k_S1_S0,1.000000e+09,fluorescent emission,FLU,True
2,k_S1_T1,1.000000e+06,intersystem crossing ST,ISCST,False
3,k_T1_S0,5.000000e+05,intersystem crossing TS,ISCTS,False
4,k_S1_S0,1.000000e+09,internal conversion S,ICS,False


In [56]:
system.joined_states

,name,single_states,single_state_counter,absorbing
id,,,,
0,S0_S0,"[0, 0]","[2, 0, 0, 0]",False
1,S0_S1,"[0, 1]","[1, 1, 0, 0]",False
2,S0_T1,"[0, 2]","[1, 0, 1, 0]",False
3,S0_B,"[0, 3]","[1, 0, 0, 1]",False
4,S1_S0,"[1, 0]","[1, 1, 0, 0]",False
5,S1_S1,"[1, 1]","[0, 2, 0, 0]",False
6,S1_T1,"[1, 2]","[0, 1, 1, 0]",False
7,S1_B,"[1, 3]","[0, 1, 0, 1]",False
8,T1_S0,"[2, 0]","[1, 0, 1, 0]",False


In [57]:
system.joined_transitions

,name,joined_states_id,single_transition_id,rate,trivial_name,fluorescence
id,,,,,,
0,S0_S0__S0_S1,"(0, 1)",0,7.000000e+06,excitation,False
1,S0_S0__S1_S0,"(0, 4)",0,7.000000e+06,excitation,False
2,S0_S1__S1_S1,"(1, 5)",0,7.000000e+06,excitation,False
3,S0_T1__S1_T1,"(2, 6)",0,7.000000e+06,excitation,False
4,S0_B__S1_B,"(3, 7)",0,7.000000e+06,excitation,False
5,S1_S0__S1_S1,"(4, 5)",0,7.000000e+06,excitation,False
6,T1_S0__T1_S1,"(8, 9)",0,7.000000e+06,excitation,False
7,B_S0__B_S1,"(12, 13)",0,7.000000e+06,excitation,False
8,S0_S1__S0_S0,"(1, 0)",1,1.000000e+09,fluorescent emission,True


In [58]:
system.transition_matrix

array([[0.00000000e+00, 5.00000000e-01, 0.00000000e+00, 0.00000000e+00,
        5.00000000e-01, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [9.96015936e-01, 0.00000000e+00, 4.98007968e-04, 0.00000000e+00,
        0.00000000e+00, 3.48605578e-03, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [6.66666667e-02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 9.33333333e-01, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.000

In [59]:
system.row_sums

array([1.4000e+07, 2.0080e+09, 7.5000e+06, 7.0000e+06, 2.0080e+09,
       4.0020e+09, 2.0015e+09, 2.0010e+09, 7.5000e+06, 2.0015e+09,
       1.0000e+06, 5.0000e+05, 7.0000e+06, 2.0010e+09, 5.0000e+05,
       0.0000e+00])

In [60]:
system.graph

In [61]:
system.plot

## Method simulate()

In [62]:
system.simulate(n_steps=50, seed=100)

In [63]:
system.time_series

array([0.00000000e+00, 1.28818253e-08, 1.35002525e-08, 1.54073146e-08,
       1.55245403e-08, 4.22203959e-08, 4.22297060e-08, 7.53146935e-08,
       7.55691615e-08, 7.58714356e-08, 7.60009364e-08, 7.61672375e-08,
       7.66316306e-08, 8.72238062e-08, 8.74075673e-08, 1.54708349e-07,
       1.54942018e-07, 2.47266172e-07, 2.49368533e-07, 4.36908655e-07,
       4.37209516e-07, 4.40425241e-07, 4.40765767e-07, 4.70775973e-07,
       4.71941597e-07, 5.09682074e-07, 5.09938845e-07, 5.51348249e-07,
       5.51618680e-07, 5.76120623e-07, 5.76680556e-07, 5.97417266e-07,
       5.97797200e-07, 6.56099154e-07, 6.56693827e-07, 6.81497196e-07,
       6.82010383e-07, 7.30075180e-07, 7.30626802e-07, 7.60004948e-07,
       7.60280401e-07, 7.78034480e-07, 7.78671437e-07, 8.01160430e-07,
       8.02001825e-07, 8.21571376e-07, 8.21574148e-07, 8.37015148e-07,
       8.37028812e-07, 8.47748614e-07, 8.48368095e-07])

In [64]:
system.time_step_series

array([0.00000000e+00, 1.28818253e-08, 6.18427249e-10, 1.90706206e-09,
       1.17225732e-10, 2.66958556e-08, 9.31010013e-12, 3.30849875e-08,
       2.54468051e-10, 3.02274099e-10, 1.29500809e-10, 1.66301073e-10,
       4.64393080e-10, 1.05921756e-08, 1.83761179e-10, 6.73007818e-08,
       2.33669306e-10, 9.23241540e-08, 2.10236041e-09, 1.87540122e-07,
       3.00860827e-10, 3.21572518e-09, 3.40526067e-10, 3.00102055e-08,
       1.16562426e-09, 3.77404773e-08, 2.56770348e-10, 4.14094045e-08,
       2.70430706e-10, 2.45019426e-08, 5.59933145e-10, 2.07367101e-08,
       3.79934610e-10, 5.83019541e-08, 5.94672603e-10, 2.48033693e-08,
       5.13186613e-10, 4.80647971e-08, 5.51621939e-10, 2.93781463e-08,
       2.75452978e-10, 1.77540783e-08, 6.36957187e-10, 2.24889933e-08,
       8.41395155e-10, 1.95695504e-08, 2.77222776e-12, 1.54410005e-08,
       1.36635957e-11, 1.07198020e-08, 6.19480869e-10])

In [65]:
system.state_series

array([0, 4, 0, 4, 0, 1, 0, 4, 0, 4, 0, 4, 0, 4, 0, 1, 0, 4, 0, 4, 0, 1,
       0, 1, 0, 4, 0, 4, 0, 1, 0, 4, 0, 1, 0, 4, 0, 4, 0, 1, 0, 1, 0, 4,
       0, 4, 0, 4, 0, 4, 0], dtype=int64)

## Method process()
Analysis of complete simulation.

In [28]:
system.single_state_series

array([[0., 1., 0., 1., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1., 0., 0.,
        0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1.,
        0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1.,
        0., 1., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
        0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.]])

In [29]:
system.transition_series

array([0, 4, 0, 1, 0, 4, 0, 4, 0, 1, 0, 1, 0, 4, 0, 4, 0, 4, 0, 1, 0, 4,
       0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 4, 0, 4, 0, 4, 0, 1, 0, 1, 0, 4,
       0, 1, 0, 4, 0, 4], dtype=int64)

In [30]:
system.single_state_lifetimes

{'single_state_occurrences': [array([0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
         1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
         0.]),
  array([0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.])],
 'single_state_occurrences_all': array([0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
        1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0.]),
 'lifetimes_fluorophore': [array([1.28818253e-08, 6.18427249e-10, 1.90706206e-09, 1.17225732e-10,
         5.97901532e-08, 2.54468051e-10, 3.02274099e-10, 1.29500809e-10,
         1.66301073e-10, 4.64393080e-10, 1.05921756e-08, 1.83761179e-10,
         1.59858605e-07, 2.10236041e-09, 1.87540122e-07, 3.00860827e-10,
         7.24725583e-08, 2.56770348e-10, 4.14094045e-08, 2.70430706e-10,
         4.57985859e-08, 3.79934610e-10, 8.36999959e-08, 5.

In [31]:
system.transition_lifetimes

{'transition_occurrences': [array([0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 4, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1,
         0, 4, 0, 4, 0, 4, 0, 1, 0, 4, 0, 4], dtype=int64),
  array([0, 4, 0, 4, 0, 4, 0, 1, 0, 1, 0, 4, 0, 1, 0, 1], dtype=int64)],
 'transition_occurrences_all': array([0, 4, 0, 1, 0, 4, 0, 4, 0, 1, 0, 1, 0, 4, 0, 4, 0, 4, 0, 1, 0, 4,
        0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 4, 0, 4, 0, 4, 0, 1, 0, 1, 0, 4,
        0, 1, 0, 4, 0, 4], dtype=int64),
 'transition_times': [[array([1.28818253e-08, 1.90706206e-09, 5.97901532e-08, 3.02274099e-10,
          1.66301073e-10, 1.05921756e-08, 1.59858605e-07, 1.87540122e-07,
          7.24725583e-08, 4.14094045e-08, 4.57985859e-08, 8.36999959e-08,
          4.80647971e-08, 7.05336280e-08, 1.95695504e-08, 1.54410005e-08,
          1.07198020e-08]),
   array([1.17225732e-10, 1.29500809e-10, 4.64393080e-10, 3.00860827e-10,
          2.70430706e-10, 3.79934610e-10, 2.77222776e-12]),
   array([], dtype=float64),
   array([], dtype=float64),
   array([6

## Method emitters()
Analysis only using events. 

In [38]:
system.emitters(photon_collection_rate=0.9, resample="5ms", emccd_gain=10)

In [41]:
system.emissions

array([ 4, 10, 12, 20, 24, 28, 30, 32, 40, 42, 46], dtype=int64)

In [42]:
system.events

array([ 4, 10, 12, 20, 24, 28, 30, 32, 42, 46], dtype=int64)

In [43]:
system.time_points_events

array([1.55245403e-08, 7.60009364e-08, 7.66316306e-08, 4.37209516e-07,
       4.71941597e-07, 5.51618680e-07, 5.76680556e-07, 5.97797200e-07,
       7.78671437e-07, 8.21574148e-07])

In [44]:
system.event_time_series

0.0    89.570646
dtype: float64

In [45]:
system.on_periods

array([1])

In [46]:
system.off_periods

array([], dtype=int32)

In [47]:
system.emission_statistics

{'total_emissions': 11,
 'total_events': 10,
 'mean_time_between_events': 8.956106750015354e-08}

## Method fcs()

In [48]:
system.fcs(normalize=True, log=True, m=2, deltat=None)

C:\Users\SagixOffice\OneDrive - Universität Würzburg\GitHub\Photoswitching\src\fluorophore_systems.py:280: PerformanceWarning: indexing past lexsort depth may impact performance.
  deltat = self.parameter_collection.loc[('emitters', 'resample'), 0]
C:\Users\SagixOffice\OneDrive - Universität Würzburg\GitHub\Photoswitching\src\fluorophore_systems.py:286: PerformanceWarning: indexing past lexsort depth may impact performance.
  if deltat == self.parameter_collection.loc[('emitters', 'resample'), 0]:


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

## Parameters

In [223]:
system.parameter_collection

0
init     number_fluorophores                                                        2
         distances                                                                  1
         single_states                                                (S0, S1, T1, B)
         rates                      [[k_S0_S1, 7000000.0, excitation, False], [k_S...
simulate n_steps                                                                   50
         seed                                                                     100
process  seed                                                                     100
emitters photon_collection_rate                                                   0.5
         resample                                                                 5ms
         emccd_gain                                                                10
         threshold                                                                  0
         memory                                                                     0
         remove_heading_off_period                                              False
         seed                                                                     100
fcs      normalize                                                               True
         log                                                                     True
         m                                                                          2
         deltat                                                                   5ms

## Time complexity

In [230]:
mi.time_complexity(mode='number_fluorophores', number_fluorophores=4, simulation_steps_exp=7, seed=100, rates=rates)

[75.06217551231384, 75.10984396934509, 67.77677512168884, 70.43964624404907]

Suppose there is an absorbing photophysical state 'B'. The simulation works with joined states, i.e., the absorbing state with 2 fluorophores would be B_B, with three fluorophores B_B_B and so on. The number of simulation steps needed to reach this state scales linearily, meaning that a system of 4 fluorophores needs on average 2 times as many steps as a system of 2 fluorophores to reach this state. \
The time complexity measurement shows that for a fixed amount of simulation steps, the number of fluorophores $n$ does not influence the time. Hence, if the goal is to reach the absorbing state of a system, the time complexity is $\mathcal{O}(n)$.

In [ ]:
mi.time_complexity(mode='simulation_steps', number_fluorophores=2, simulation_steps_exp=8, seed=10, rates=rates)

[0.04687213897705078,
 0.10647034645080566,
 0.6981048583984375,
 7.010973930358887,
 71.38469338417053,
 840.2985830307007]

Suppose $n$ is the number of simulation steps, then the time complexity is $\mathcal{O}(n)$. 